In [1]:
!pip install \
    --no-index \
    --find-links=/kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels \
    vllm==0.10.1.1 \
    transformers==4.56.1

# dense_rag.py needs faiss; best-effort so a missing wheel cannot kill the
# vllm install above (the fallback just stays off without it).
!pip install --no-index \
    --find-links=/kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels \
    faiss-cpu \
    || echo "faiss-cpu wheel not attached - dense RAG fallback stays off"

Looking in links: /kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels
Processing /kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels/vllm-0.10.1.1-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels/transformers-4.56.1-py3-none-any.whl
Processing /kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels/prometheus_fastapi_instrumentator-8.0.2-py3-none-any.whl (from vllm==0.10.1.1)
Processing /kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels/lm_format_enforcer-0.10.12-py3-none-any.whl (from vllm==0.10.1.1)
Processing /kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels/llguidance-0.7.30-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from vllm==0.10.1.1)
Processing /kaggle/input/notebooks/siamahmad/bn-halucination-utils/wheels/outlines_core-0.2.10-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (from vllm==0.10.1.1)
Processing /kaggle/input/notebooks/siamahmad/bn-

In [2]:
%%writefile prep_data.py
import json
import os

import pandas as pd

# Bengali -> English digit mapping (worth +0.007 on the LB in v1 — keep it)
BN_TO_EN = str.maketrans({
    "০": "0", "১": "1", "২": "2", "৩": "3", "৪": "4",
    "৫": "5", "৬": "6", "৭": "7", "৮": "8", "৯": "9",
})

TEXT_COLS = ["context", "prompt_bn", "response_bn"]

# Question-type patterns used only to pick a diverse few-shot set
WM_RE = "ভাবার্থ|শাব্দিক অর্থ|বাগধারা|প্রবাদ"
MATH_RE = "অনুপাত|শতকরা|সুদ|গড়|লিটার|ত্রিভুজ|বর্গ|সংখ্যাটি|কত দিনে|লাভ|ক্ষতি|সমীকরণ|গ.সা.গু|ল.সা.গু"


def find_data_dir():
    here = os.path.dirname(os.path.abspath(__file__))
    for d in ("/kaggle/input/competitions/bengali-hallucination",
              "/kaggle/input/bengali-hallucination",
              here):
        if os.path.exists(os.path.join(d, "test set.csv")):
            return d
    # phase-2 rerun may mount the data elsewhere — search everything attached
    import glob
    for hit in glob.glob("/kaggle/input/**/test set.csv", recursive=True):
        return os.path.dirname(hit)
    raise FileNotFoundError("competition files not found in any known location")


def convert_digits(df):
    for col in TEXT_COLS:
        df[col] = df[col].fillna("").astype(str).str.translate(BN_TO_EN)
    return df


def null_mask(df):
    # A missing/empty context is a closed-book row, same as "[NULL]"
    # (the labeled samples contain two rows with no context field at all)
    return df["context"].str.strip().isin(["", "[NULL]"])


def pick_example(df, mask, label, used):
    cand = df[mask & ~df.index.isin(used)]
    strict = cand[cand["label"] == label]
    if not strict.empty:
        cand = strict
    if cand.empty:
        return None
    length = cand["context"].str.len() + cand["prompt_bn"].str.len()
    return length.sort_values().index[0]


def main():
    data_dir = find_data_dir()
    print("data dir:", data_dir)

    test = pd.read_csv(os.path.join(data_dir, "test set.csv"))
    if "id" not in test.columns:
        test["id"] = test.index + 1
    samples = os.path.join(data_dir, "dataset samples.json")
    if not os.path.exists(samples):
        import glob
        hits = glob.glob("/kaggle/input/**/dataset samples.json", recursive=True)
        assert hits, "dataset samples.json not found in any attached input"
        samples = hits[0]
    with open(samples, encoding="utf-8") as f:
        dev = pd.json_normalize(json.load(f))
    convert_digits(test)
    convert_digits(dev)

    # Pick few-shot examples out of the labeled samples; the rest becomes the
    # local eval set. Aim for label balance and question-type coverage.
    is_ctx = ~null_mask(dev)
    q = dev["prompt_bn"]
    wm = q.str.contains(WM_RE, na=False)
    math = q.str.contains(MATH_RE, na=False) & ~wm
    gk = ~wm & ~math

    used = set()

    def take(mask, label):
        idx = pick_example(dev, mask, label, used)
        if idx is not None:
            used.add(idx)
        return idx

    ctx_idx = [take(is_ctx, 1), take(is_ctx, 0)]
    null_idx = [take(~is_ctx & wm, 1), take(~is_ctx & math, 0),
                take(~is_ctx & gk, 1), take(~is_ctx & gk, 0)]

    def row_dict(i):
        r = dev.loc[i]
        return {"context": r["context"], "prompt_bn": r["prompt_bn"],
                "response_bn": r["response_bn"], "label": int(r["label"])}

    fewshot = {"ctx": [row_dict(i) for i in ctx_idx if i is not None],
               "null": [row_dict(i) for i in null_idx if i is not None]}
    with open("fewshot.json", "w", encoding="utf-8") as f:
        json.dump(fewshot, f, ensure_ascii=False, indent=1)

    dev_eval = dev.drop(index=list(used)).reset_index(drop=True)
    dev_ctx = dev_eval[~null_mask(dev_eval)].reset_index(drop=True)
    dev_null = dev_eval[null_mask(dev_eval)].reset_index(drop=True)
    dev_ctx.to_parquet("dev_ctx.parquet", index=False)
    dev_null.to_parquet("dev_null.parquet", index=False)

    test_ctx = test[~null_mask(test)].reset_index(drop=True)
    test_null = test[null_mask(test)].reset_index(drop=True)
    test_ctx.to_parquet("test_ctx.parquet", index=False)
    test_null.to_parquet("test_null.parquet", index=False)

    print(f"few-shot: ctx={len(fewshot['ctx'])} null={len(fewshot['null'])}")
    print(f"dev eval: ctx={len(dev_ctx)} null={len(dev_null)}")
    print(f"test:     ctx={len(test_ctx)} null={len(test_null)} "
          f"ids {test['id'].min()}..{test['id'].max()}")


if __name__ == "__main__":
    main()


Writing prep_data.py


In [3]:
%%writefile judge.py
import os
import re
import time
from collections import Counter

import pandas as pd

# Must be set before vllm is imported (vllm import is deferred to load_model):
# force the legacy V0 engine — V1 refuses sm_75 (T4) — and use the xformers
# attention path since Turing has no FlashAttention 2.
os.environ.setdefault("VLLM_USE_V1", "0")
os.environ.setdefault("VLLM_ATTENTION_BACKEND", "XFORMERS")

# Pre-quantized AWQ build (~17 GB) — full fp16 is ~60 GB and won't fit on 2x T4.
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/32b-awq/1"
# MODEL_PATH = "/kaggle/input/models/konstantinboyko/deepseek-r1-distill-qwen-32b-awq-casperhansen/transformers/default/1"



def find_model_dir():
    """Locate the Qwen3 AWQ checkpoint wherever it is mounted.

    The official qwen-lm/qwen-3 model and community re-uploads mount at
    different /kaggle/input paths, so a hardcoded path breaks as soon as a
    different (but equivalent) model input is attached — including in the
    hosts' rerun. Prefer MODEL_PATH, else scan for a qwen3 config.json with
    weights next to it, preferring AWQ-quantized builds (fp16 won't fit 2xT4).
    """
    import glob
    import json

    if os.path.isfile(os.path.join(MODEL_PATH, "config.json")):
        return MODEL_PATH
    fallback = None
    for cfg in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        d = os.path.dirname(cfg)
        try:
            with open(cfg) as f:
                c = json.load(f)
        except (OSError, ValueError):
            continue
        if "qwen3" not in str(c.get("model_type", "")).lower():
            continue
        if not glob.glob(os.path.join(d, "*.safetensors")):
            continue
        quant = str((c.get("quantization_config") or {}).get("quant_method", ""))
        if quant.lower() == "awq":
            return d
        fallback = fallback or d
    if fallback:
        return fallback
    raise FileNotFoundError(
        "No Qwen3 checkpoint under /kaggle/input — attach the model "
        "'qwen-lm/qwen-3' (Transformers, variation 32b-awq) as a notebook input.")

# Worst-case Bengali context (~7.2k chars) tokenizes to roughly 6k tokens; add
# system prompt + few-shot + 3k thinking budget and 8192 no longer fits.
MAX_MODEL_LEN = 12288
MAX_GEN_TOKENS = 3000  # thinking budget: rows judged from model knowledge
FAST_MAX_TOKENS = 768  # referenced rows: direct comparison, no thinking

# Self-consistency votes per branch. Measured on dev: majority-of-3 on the
# null branch scored 0.7868 vs 0.7847 for a single vote (fixed 8 rows, broke
# 7) — not worth ~2x runtime, and the rerun on the hidden test set must not
# risk the 12 h limit. Single vote everywhere.
CTX_VOTES = 1
NULL_VOTES = 1

SYSTEM_CTX = """You are an extremely strict verifier.

Your task is to decide whether the Candidate Answer correctly answers the Question using the provided Context.

You receive both Bangla and English versions.

SOURCE OF TRUTH

• The Bangla Context is authoritative.
• The English Context is only a translation aid.
• If the Bangla and English texts differ, ALWAYS trust the Bangla text.
• Use the Context as the primary evidence when it contains the relevant information.

EVALUATION RULES

A candidate answer is CORRECT only if it answers the exact question being asked.

Before deciding, identify exactly what information the question requests
(person, organization, place, country, date, year, number, reason, result,
title, language, inventor, founder, winner, first, second, last, before,
after, largest, smallest).

Many contexts contain multiple people, dates, places, numbers and events.

An answer is INCORRECT if it gives a true fact from the Context that does NOT answer the requested question.

Examples:

Question asks WHERE — Answer gives WHEN → Incorrect
Question asks FIRST — Answer gives SECOND → Incorrect
Question asks PERSON — Answer gives ORGANIZATION → Incorrect
Question asks DIRECTOR — Answer gives PRODUCER → Incorrect

NUMBER CHECK

Compare every number DIGIT BY DIGIT. 249 is NOT 269. 1907 is NOT 1909.
A close-but-different number is INCORRECT.

DATE GRANULARITY

If the question asks when (কবে / কখন / কোন সালে) and the Candidate gives a
year (or year range) that is consistent with the Context, treat it as CORRECT
even if the Context gives a fuller date.

ENTITY MATCHING

Treat the following as equivalent when they clearly refer to the same entity:

• honorifics and titles (বেগম রোকেয়া = রোকেয়া)
• spelling variations and transliterations
• Bangla term vs its English/loanword equivalent (আস্তিক = থিয়িস্ট)
• a name with one extra or missing middle word, if it is clearly the same person
• an alternative well-known name of the same place or thing
  (রবীন্দ্র সেতু = হাওড়া ব্রিজ)

Do NOT reject an answer because of titles, honorifics, or extra correct detail.

ORDINAL REASONING

Be especially careful with: first, second, third, last, before, after,
previous, next, earliest, latest.
Trace the complete sequence before making a decision.
Do not match only the first similar name that appears.

MULTI-STEP REASONING

Information may be spread across multiple sentences. Combine evidence when necessary.

CONTEXT INSUFFICIENT — IMPORTANT

The Context is often a truncated snippet and may NOT contain the fact being asked.
If the Context does not contain the information needed to answer the Question:

• Do NOT return 0 just because the Context is silent.
• Instead, answer the Question from your own knowledge.
• If the Candidate Answer matches what you know to be correct → return 1.
• If the Candidate Answer contradicts what you know, or you know a different
  correct answer → return 0.

Only treat the Context as overriding your knowledge when the Context actually
contains the relevant information.

FINAL DECISION

Return 1 if:
• the Context supports the answer, OR
• the Context is silent on the question and the answer is factually correct
  according to your own knowledge.

Return 0 if:
• the answer names the wrong entity / attribute / number, OR
• the Context (or, when the Context is silent, your own knowledge) contradicts it.

Output EXACTLY this format.

VERDICT: \\boxed{1}

or

VERDICT: \\boxed{0}

Output nothing else."""

SYSTEM_NULL = """You are an expert fact-checker for Bengali (Bangla) question-answer pairs.

You will be given a Question and a Candidate Answer in Bengali, with no reference context.
Decide whether the Candidate Answer is factually correct.

Procedure:
1. First, work out the correct answer yourself.
   - Math / arithmetic word problems: solve the problem step by step and compute the result yourself before judging.
   - Bengali grammar and word-meaning questions: recall the standard grammar-book answer.
   - General-knowledge questions: recall the relevant facts.
2. Compare with the Candidate Answer:
   - Matches the correct answer (minor wording, spelling or digit-script differences are fine) -> \\boxed{1}
   - Factually wrong, contradicts the correct answer, off-topic, or does not answer the question -> \\boxed{0}
   - You genuinely cannot determine the correct answer -> \\boxed{-1}

Rules:
- Judge factual correctness only; ignore tone, fluency and grammar.
- A fluent, confident answer is still \\boxed{0} if its facts are wrong.
- A REFERENCE section may quote an answer key from a public exam question bank or a dictionary entry for the asked idiom. When present, trust it over your own recall: label 1 if the Candidate Answer agrees in meaning with the referenced correct answer, label 0 if it does not. Never answer \\boxed{-1} when a REFERENCE is present.
- End your response with exactly one label: \\boxed{1}, \\boxed{0} or \\boxed{-1}. Write nothing after the label."""

BOX_RE = re.compile(r"\\boxed\{\s*(-1|0|1)\s*\}")


def load_model():
    from vllm import LLM
    model_dir = find_model_dir()
    print(f"loading model from: {model_dir}")
    llm = LLM(
        model=model_dir,
        dtype="float16",
        tensor_parallel_size=2,
        gpu_memory_utilization=0.90,
        max_model_len=MAX_MODEL_LEN,
        enforce_eager=True,
        trust_remote_code=True,
    )
    return llm, llm.get_tokenizer()


def make_sampling(n, thinking=True):
    from vllm import SamplingParams
    # Thinking mode: greedy decoding is explicitly discouraged for Qwen3
    # (repetition loops); recommended temp 0.6, top_p 0.95, top_k 20.
    # Non-thinking mode (referenced rows): recommended temp 0.7, top_p 0.8.
    max_tokens = MAX_GEN_TOKENS if thinking else FAST_MAX_TOKENS
    return SamplingParams(
        temperature=0.6 if thinking else 0.7,
        top_p=0.95 if thinking else 0.8,
        top_k=20,
        n=n,
        seed=1234,
        max_tokens=max_tokens,
        truncate_prompt_tokens=MAX_MODEL_LEN - max_tokens - 64,
    )


def fewshot_block(examples, with_context):
    lines = ["", "", "Here are solved examples:"]
    for i, ex in enumerate(examples, 1):
        lines.append(f"\nExample {i}:")
        if with_context:
            lines.append(f"Context:\n{ex['context']}")
        lines.append(f"Question:\n{ex['prompt_bn']}")
        lines.append(f"Candidate Answer:\n{ex['response_bn']}")
        lines.append(f"Correct label: \\boxed{{{ex['label']}}}")
    return "\n".join(lines)


def user_ctx(row, ref=None):
    parts = [f"Context:\n{row['context']}",
             f"Question:\n{row['prompt_bn']}",
             f"Candidate Answer:\n{row['response_bn']}"]
    if ref:
        parts.append(ref)
        parts.append("Judge the Candidate Answer against the REFERENCE "
                     "answer(s). Think carefully, then end with \\boxed{1} "
                     "or \\boxed{0}.")
    else:
        parts.append("Is the Candidate Answer fully supported by the Context? "
                     "Think carefully, then end with \\boxed{1} or \\boxed{0}.")
    return "\n\n".join(parts)


def user_null(row, ref=None):
    parts = [f"Question:\n{row['prompt_bn']}",
             f"Candidate Answer:\n{row['response_bn']}"]
    if ref:
        parts.append(ref)
        parts.append("Is the Candidate Answer factually correct? Use the "
                     "REFERENCE, then end with \\boxed{1} or \\boxed{0}.")
    else:
        parts.append("Is the Candidate Answer factually correct? Work out the "
                     "correct answer yourself first, then end with \\boxed{1}, "
                     "\\boxed{0} or \\boxed{-1}.")
    return "\n\n".join(parts)


def parse_label(text, allow_abstain):
    # Prefer the answer after the thinking block; fall back to the whole text.
    tail = text.split("</think>")[-1]
    matches = BOX_RE.findall(tail) or BOX_RE.findall(text)
    for m in reversed(matches):
        v = int(m)
        if v != -1 or allow_abstain:
            return v
    return None


def aggregate(votes, allow_abstain):
    votes = [v for v in votes if v is not None]
    if not votes:
        return -1 if allow_abstain else 0
    n = len(votes)
    counts = Counter(votes)
    if counts.get(1, 0) * 2 > n:
        return 1
    if counts.get(0, 0) * 2 > n:
        return 0
    return -1 if allow_abstain else 0


def run_branch(llm, tokenizer, df, branch, fewshot, n_votes, refs=None,
               deadline=None):
    """Judge every row of df.

    Rows with a retrieved REFERENCE are judged in non-thinking mode with a
    short token budget — the task is a direct comparison, and this runs ~5-10x
    faster. Unreferenced rows get the full thinking budget, in chunks; if
    `deadline` (epoch seconds) passes between chunks, the remaining rows fall
    back to fast mode as well — a timed-out rerun scores zero, a degraded one
    does not.
    """
    allow_abstain = branch == "null"
    base = SYSTEM_NULL if allow_abstain else SYSTEM_CTX
    system = base + fewshot_block(fewshot, with_context=not allow_abstain)
    build_user = user_null if allow_abstain else user_ctx
    refs = refs if refs is not None else [None] * len(df)

    rows = [row for _, row in df.iterrows()]
    outputs = [None] * len(rows)

    def generate(idx, thinking):
        if not idx:
            return
        t0 = time.time()
        prompts = []
        for i in idx:
            ref_text = refs[i]["text"] if refs[i] else None
            messages = [{"role": "system", "content": system},
                        {"role": "user", "content": build_user(rows[i], ref_text)}]
            prompts.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=thinking))
        outs = llm.generate(prompts, make_sampling(n_votes, thinking))
        for i, o in zip(idx, outs):
            outputs[i] = o
        dt = time.time() - t0
        print(f"[{branch}] {'think' if thinking else 'fast'}: {len(idx)} rows "
              f"in {dt:.0f}s ({dt / len(idx):.2f} s/row)", flush=True)

    fast = [i for i, r in enumerate(refs) if r is not None and r["fast"]]
    slow = [i for i, r in enumerate(refs) if r is None or not r["fast"]]
    generate(fast, thinking=False)
    if deadline is None:
        generate(slow, thinking=True)
    else:
        chunk = 400
        for start in range(0, len(slow), chunk):
            if time.time() > deadline:
                print(f"[{branch}] time budget exceeded — judging remaining "
                      f"{len(slow) - start} rows in fast mode", flush=True)
                generate(slow[start:], thinking=False)
                break
            generate(slow[start:start + chunk], thinking=True)

    finals, debug, truncated, unparsed = [], [], 0, 0
    for out in outputs:
        votes = []
        for gen in out.outputs:
            if gen.finish_reason == "length":
                truncated += 1
            label = parse_label(gen.text, allow_abstain)
            if label is None:
                unparsed += 1
            votes.append(label)
        finals.append(aggregate(votes, allow_abstain))
        debug.append({"votes": str(votes),
                      "tail": out.outputs[0].text.split("</think>")[-1].strip()[-200:]})
    total_gens = len(df) * n_votes
    print(f"[{branch}] rows={len(df)} gens={total_gens} "
          f"truncated={truncated} unparsed={unparsed}")
    return finals, pd.DataFrame(debug)


def macro_f1(y_true, y_pred):
    f1s = []
    for c in (0, 1):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        f1s.append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0.0)
    return sum(f1s) / len(f1s)


def choose_mapping(ctx_true, ctx_pred, null_true, null_raw):
    ctx_f1 = macro_f1(ctx_true, ctx_pred)
    abstains = sum(1 for v in null_raw if v == -1)
    lines = [f"context branch macro F1: {ctx_f1:.4f}  (n={len(ctx_true)})",
             f"null branch abstains: {abstains}/{len(null_raw)}"]
    best_m, best_f1 = 0, -1.0
    for m in (0, 1):
        null_pred = [m if v == -1 else v for v in null_raw]
        null_f1 = macro_f1(null_true, null_pred)
        overall = macro_f1(ctx_true + null_true, ctx_pred + null_pred)
        lines.append(f"map -1 -> {m}: null macro F1 {null_f1:.4f}, "
                     f"overall macro F1 {overall:.4f}")
        if overall > best_f1:
            best_m, best_f1 = m, overall
    lines.append(f"chosen: -1 -> {best_m} (local overall macro F1 {best_f1:.4f})")
    return best_m, "\n".join(lines)


Writing judge.py


In [ ]:
%%writefile retrieval.py

"""General retrieval over a question-answer bank (RAG for the judge).

Any .csv / .parquet file that has a question column and an answer column is
loaded into one bank. Drop a new file into the data folder (or attach
another Kaggle dataset) and it is picked up automatically — no code change.
Files are recognized by SCHEMA, not by filename, so data1.csv, bcs_gk.csv,
anything.parquet all work. Competition/output files (prompt_bn/response_bn/
label columns) are excluded automatically.

Matching, per row of the judge input:
1. exact — normalized-key lookup (Bijoy াে→ো fix, Bengali→ASCII digits,
   punctuation stripped). Single agreed answer -> fast reference.
2. law — a question citing an act + year + ধারা gets the statute text
   quoted from a legal-sections corpus (act_title/act_year/section_no/text
   schema, e.g. legal_sections.parquet), also discovered by schema.
3. fuzzy — TF-IDF over character 2-4 grams, top-3 above MIN_SIM.
4. rule tables — সমাস / সন্ধি grammar rules for rows the bank misses.
5. dense fallback — rows all of the above miss are handed to dense_rag.py
   (BGE-M3 claim embedding over the encyclopedia FAISS index built by the
   rag-bangla notebook); top passages above a cosine floor are quoted as
   evidence. A printed no-op unless that index + model are attached.

The judge model still makes every prediction itself; retrieval only adds
reference context to the prompt. With no data attached this is a no-op.

Self-test: `python3 retrieval.py` (uses ./filter_data or $DATA_DIR).
"""
import glob
import os
import re
from collections import Counter

import numpy as np
import pandas as pd

# ---------------------------------------------------------------- config
QUESTION_COLS = ("question", "questions", "prompt", "query")
ANSWER_COLS = ("answer", "answers", "answer_text", "ans")
EXCLUDE_COLS = {"prompt_bn", "response_bn", "label", "prediction"}
LAW_COLS = {"act_title", "act_year", "section_no", "text"}
MIN_SIM = 0.55   # fuzzy-match floor (validated on dev in earlier runs)
TOP_K = 3

# ---------------------------------------------------------- normalization
BN_DIGITS = "০১২৩৪৫৬৭৮৯"
_DIGIT_TBL = {ord(b): str(i) for i, b in enumerate(BN_DIGITS)}
# Bijoy-encoded text stores broken two-glyph vowel signs; both orders occur:
# \u09be+\u09c7 -> \u09cb (o-kar), \u09c7+\u09d7 -> \u09cc (ou-kar)
_BIJOY = (("\u09be\u09c7", "\u09cb"), ("\u09c7\u09be", "\u09cb"),
          ("\u09c7\u09d7", "\u09cc"), ("\u09d7\u09c7", "\u09cc"))


def clean(s):
    """Display-level normalization: Bijoy fix + whitespace collapse."""
    s = str(s)
    for broken, fixed in _BIJOY:
        s = s.replace(broken, fixed)
    return re.sub(r"\s+", " ", s).strip()


def soft(s):
    """Fuzzy-index text: cleaned, ASCII digits, lowercased."""
    return clean(s).translate(_DIGIT_TBL).lower()


def hard(s):
    """Exact-match key: soft() with all non-alphanumerics stripped."""
    if not isinstance(s, str):
        return ""
    return re.sub(r"[\W_]+", "", soft(s), flags=re.UNICODE)


_TYPES = ("ভাবার্থ", "শাব্দিক")


def meaning_type(q):
    """Partition idiom questions: ভাবার্থ queries must not be answered with
    শাব্দিক entries and vice versa (the two meanings differ)."""
    for t in _TYPES:
        if t in q:
            return t
    return ""


# ------------------------------------------------------------- discovery
# the attached Kaggle dataset holding the question bank (searched first;
# a recursive /kaggle/input scan still picks up any other attached data)
KAGGLE_DATASET = ("/kaggle/input/datasets/imdadulhasanhamim/"
                  "bangla-helucination-qn-pair")
def _columns(path):
    try:
        if path.endswith(".csv"):
            return list(pd.read_csv(path, nrows=0).columns)
        import pyarrow.parquet as pq
        return list(pq.read_schema(path).names)
    except Exception as e:
        print(f"[rag] skipping unreadable {path}: {e}")
        return None


def find_data_files(roots=None):
    """Scan roots for bank files (by schema). Returns (qa_files, law_files)
    where qa_files is [(path, question_col, answer_col), ...]."""
    if roots is None:
        env = [os.environ[k] for k in ("DATA_DIR", "SOURCES_DIR")
               if os.environ.get(k)]
        here = os.path.dirname(os.path.abspath(__file__))
        roots = env or [KAGGLE_DATASET, "/kaggle/input",
                        os.path.join(here, "filter_data"),
                        os.path.join(os.getcwd(), "filter_data")]
    qa, law, seen = [], [], set()
    for root in roots:
        for p in sorted(glob.glob(os.path.join(root, "**", "*.*"),
                                  recursive=True)):
            rp = os.path.realpath(p)
            if not p.endswith((".csv", ".parquet")) or rp in seen:
                continue
            seen.add(rp)
            cols = _columns(p)
            if cols is None:
                continue
            low = {c.lower().strip(): c for c in cols}
            if EXCLUDE_COLS & low.keys():
                continue
            if LAW_COLS <= low.keys():
                law.append(p)
                continue
            qcol = next((low[c] for c in QUESTION_COLS if c in low), None)
            acol = next((low[c] for c in ANSWER_COLS if c in low), None)
            if qcol and acol:
                qa.append((p, qcol, acol))
    return qa, law


# ------------------------------------------------------------------ bank
class Bank:
    """All QA files merged into one index, deduped on (question, answer)."""

    def __init__(self, qa_files):
        self.qs, self.answers, self.types = [], [], []
        index = {}
        for path, qcol, acol in qa_files:
            name = os.path.splitext(os.path.basename(path))[0]
            try:
                if path.endswith(".csv"):
                    df = pd.read_csv(path, usecols=[qcol, acol])
                else:
                    df = pd.read_parquet(path, columns=[qcol, acol])
            except Exception as e:
                print(f"[rag] skipping {name}: {e}")
                continue
            kept = 0
            for q, a in zip(df[qcol], df[acol]):
                q, a = clean(q), clean(a)
                hq, ha = hard(q), hard(a)
                if not hq or not ha:
                    continue
                i = index.get(hq)
                if i is None:
                    i = index[hq] = len(self.qs)
                    self.qs.append(q)
                    self.answers.append({})
                    self.types.append(meaning_type(q))
                self.answers[i].setdefault(ha, (a, set()))[1].add(name)
                kept += 1
            print(f"[rag] loaded {name}: {kept}/{len(df)} usable rows")
        self.exact = index
        self._types = np.array(self.types)
        self._vec = self._mat = None
        print(f"[rag] bank: {len(self.qs)} unique questions "
              f"from {len(qa_files)} files")

    def _build_fuzzy(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        self._vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4),
                                    sublinear_tf=True)
        self._mat = self._vec.fit_transform(soft(q) for q in self.qs)

    def _entry(self, i, sim, is_exact):
        answers = [(a, sorted(srcs)) for a, srcs in self.answers[i].values()]
        return {"question": self.qs[i], "answers": answers,
                "sim": sim, "exact": is_exact}

    def query(self, question, k=TOP_K, min_sim=MIN_SIM):
        """Top-k bank entries for a question; exact hit (if any) is first."""
        q = clean(str(question))
        hits, skip = [], -1
        i = self.exact.get(hard(q))
        if i is not None:
            hits.append(self._entry(i, 1.0, True))
            skip = i
        if len(hits) >= k or not self.qs:
            return hits[:k]
        if self._vec is None:
            self._build_fuzzy()
        sims = (self._mat @ self._vec.transform([soft(q)]).T).toarray().ravel()
        qt = meaning_type(q)
        if qt:
            sims[self._types != qt] = 0.0
        if skip >= 0:
            sims[skip] = 0.0
        for j in np.argsort(-sims)[:k]:
            if sims[j] < min_sim or len(hits) >= k:
                break
            hits.append(self._entry(int(j), float(sims[j]), False))
        return hits


# ------------------------------------------------------------ law corpus
# "ডিএনএ আইন, 2014 এর 28 ধারা ..." — act name + year, then cited sections.
# Bengali titles end with আইন/অধ্যাদেশ; English ones with Act/Ordinance.
_LAW_ACT = re.compile(
    r"([ঀ-৿][^।?\n]{0,70}?(?:আইন|অধ্যাদেশ)|"
    r"[A-Za-z][^।?\n]{0,70}?(?:Act|Ordinance))\s*,?\s*([0-9০-৯]{4})",
    re.UNICODE)
_LAW_SECS = re.compile(r"([0-9০-৯]+)\s*(?:নং\s*)?ধারা|ধারা\s*([0-9০-৯]+)",
                       re.UNICODE)


class LawIndex:
    def __init__(self, paths):
        self.acts = {}  # hard(title+year) -> {"title", "year", "secs"}
        for path in paths:
            for _, r in pd.read_parquet(path).iterrows():
                ent = self.acts.setdefault(
                    hard(str(r["act_title"])) + hard(str(r["act_year"])),
                    {"title": clean(r["act_title"]),
                     "year": hard(str(r["act_year"])), "secs": {}})
                try:
                    ent["secs"].setdefault(int(hard(str(r["section_no"]))),
                                           str(r["text"]))
                except ValueError:
                    pass
        print(f"[rag] law corpus: {len(self.acts)} acts")

    def reference(self, q):
        m = _LAW_ACT.search(q)
        if not m:
            return None
        name, year = hard(m.group(1)), hard(m.group(2))
        act = self.acts.get(name + year)
        if act is None:
            # shorthand vs official title: same year + containment, only
            # when unambiguous
            cands = [v for k, v in self.acts.items()
                     if v["year"] == year and (name in k or k in name + year)]
            if len(cands) != 1:
                return None
            act = cands[0]
        # only explicitly cited sections — quoting a guessed section risks
        # pointing the judge at the wrong text, which is worse than nothing
        cited = []
        for a, b in _LAW_SECS.findall(q):
            try:
                n = int(hard(a or b))
            except ValueError:
                continue
            if n in act["secs"] and n not in cited:
                cited.append(n)
        if not cited:
            return None
        quoted = "\n".join(f"ধারা {n}: {act['secs'][n][:900]}"
                           for n in cited[:3])
        return ("REFERENCE — বাংলাদেশ আইন কর্পাস থেকে "
                f"“{act['title']}”:\n{quoted}\n"
                "এই ধারার পাঠ্যের ভিত্তিতে Candidate Answer সঠিক কিনা বিচার করুন; "
                "পাঠ্যে প্রয়োজনীয় তথ্য না থাকলে নিজের জ্ঞানে বিচার করুন।")


# ------------------------------------------------- grammar rule fallbacks
# Standard NCTB grammar rules — static reference knowledge the model applies,
# same category as a dictionary entry. Need reasoning, so fast=False.
SAMAS_RULES = """REFERENCE — সমাস নির্ণয়ের প্রামাণ্য নিয়ম (ব্যাসবাক্যের গঠন দেখে):
- "X ও Y" / "X এবং Y" → দ্বন্দ্ব সমাস
- "Xের Y" → ষষ্ঠী তৎপুরুষ
- "Xে Y" / "Xতে Y" → সপ্তমী তৎপুরুষ
- "X হইতে/থেকে Y" → পঞ্চমী তৎপুরুষ
- "X দ্বারা/দিয়া/কর্তৃক Y" → তৃতীয়া তৎপুরুষ
- "Xের জন্য/নিমিত্ত Y" → চতুর্থী তৎপুরুষ
- "Xকে Y" → দ্বিতীয়া তৎপুরুষ
- "না/নয় X" → নঞ তৎপুরুষ
- "নাই/নেই X যার" → নঞ বহুব্রীহি
- ব্যাসবাক্য "যার/যাহার" দিয়ে অন্য কিছুকে বোঝালে → বহুব্রীহি
- "Xের ন্যায়/মতো Y" যেখানে Y গুণবাচক (বিশেষণ) → উপমান কর্মধারয়
- "X Yের ন্যায়" যেখানে দুটিই বিশেষ্য (X-কে Y-এর সঙ্গে তুলনা) → উপমিত কর্মধারয়
- "X-ই Y" (অভেদ কল্পনা) → রূপক কর্মধারয়
- সংখ্যাবাচক প্রথম পদ ও সমাহার অর্থ → দ্বিগু সমাস
- অব্যয় প্রথম পদ → অব্যয়ীভাব সমাস
- "যে X-ই Y" / বিশেষণ + বিশেষ্য → কর্মধারয় সমাস
ব্যাসবাক্যের গঠনে এই নিয়ম প্রয়োগ করে সঠিক সমাসটি নির্ণয় করুন, তারপর Candidate Answer সেটির সাথে মিললে 1, না মিললে 0 দিন।"""

SANDHI_RULES = """REFERENCE — স্বরসন্ধির মূল সূত্র:
- অ/আ + অ/আ = আ (দেব+আলয়=দেবালয়, বিদ্যা+আলয়=বিদ্যালয়)
- অ/আ + ই/ঈ = এ (নর+ইন্দ্র=নরেন্দ্র, মহা+ঈশ=মহেশ)
- অ/আ + উ/ঊ = ও (সূর্য+উদয়=সূর্যোদয়)
- অ/আ + ঋ = অর্ (মহা+ঋষি=মহর্ষি)
- অ/আ + এ/ঐ = ঐ; অ/আ + ও/ঔ = ঔ (মহা+ঔষধ=মহৌষধ)
- ই/ঈ + ই/ঈ = ঈ (গিরি+ইন্দ্র=গিরীন্দ্র, নদী+ঈশ=নদীশ)
- ই/ঈ + ভিন্ন স্বর = য্‌-ফলা (প্রতি+এক=প্রত্যেক, অতি+অন্ত=অত্যন্ত)
- উ/ঊ + ভিন্ন স্বর = ব্‌-ফলা যুক্ত রূপ (সু+আগত=স্বাগত)
এই সূত্র প্রয়োগ করে সঠিক সন্ধিবদ্ধ শব্দ নির্ণয় করুন; Candidate Answer সেটির সাথে মিললে 1, না মিললে 0 দিন।"""

_SAMAS_Q = re.compile("সমাস")
_SANDHI_Q = re.compile("সন্ধি")


# ------------------------------------------------------------- interface
def _bank_text(hits):
    """Reference block for bank hits -> (text, fast)."""
    top = hits[0]
    if top["exact"]:
        answers = top["answers"]
        if len(answers) == 1:
            a, srcs = answers[0]
            return ("REFERENCE — this exact Question appears in the attached "
                    f"question bank (source: {', '.join(srcs)}):\n"
                    f"Answer key: {a[:400]}", True)
        lines = "\n".join(f"- {a[:300]} [{', '.join(s)}]"
                          for a, s in answers[:3])
        return ("REFERENCE — this exact Question appears in the question "
                "bank, but its copies list different answers (the bank may "
                "contain errors; use with judgement):\n" + lines, False)
    lines = []
    for h in hits:
        a = " / ".join(a for a, _ in h["answers"][:2])
        srcs = sorted({s for _, ss in h["answers"] for s in ss})
        lines.append(f"- (similarity {h['sim']:.2f}) Q: {h['question'][:250]}\n"
                     f"  A: {a[:250]} [{', '.join(srcs)}]")
    return ("REFERENCE — similar question(s) retrieved from the question "
            "bank. Use them only if they truly answer this Question; "
            "otherwise judge from your own knowledge:\n" + "\n".join(lines),
            False)


def _reference_for(q, bank, law):
    hits = bank.query(q) if bank else []
    if hits and hits[0]["exact"]:
        text, fast = _bank_text(hits)
        return {"text": text, "fast": fast, "kind": "exact-answer-key"}
    if law:
        text = law.reference(q)
        if text:
            return {"text": text, "fast": False, "kind": "law-section"}
    if hits:
        text, fast = _bank_text(hits)
        return {"text": text, "fast": fast, "kind": "similar-question"}
    if _SAMAS_Q.search(q):
        return {"text": SAMAS_RULES, "fast": False, "kind": "rule-table"}
    if _SANDHI_Q.search(q):
        return {"text": SANDHI_RULES, "fast": False, "kind": "rule-table"}
    return None


_CACHE = {}


def _load():
    if "bank" not in _CACHE:
        qa_files, law_files = find_data_files()
        _CACHE["bank"] = Bank(qa_files) if qa_files else None
        _CACHE["law"] = LawIndex(law_files) if law_files else None
    return _CACHE["bank"], _CACHE["law"]


def build_reference(df, branch):
    """Per row of df (needs a prompt_bn column): None, or
    {"text": reference block, "fast": bool}.

    fast=True marks agreed answer-key references (a direct comparison, safe
    to judge without thinking); fast=False marks similar-question, law,
    rule-table and dense-passage references that still need reasoning.
    """
    bank, law = _load()
    if bank is None and law is None:
        print(f"[rag] no bank data found — {branch} has only the dense fallback")
        out = [None] * len(df)
    else:
        out = [_reference_for(str(r["prompt_bn"]), bank, law)
               for _, r in df.iterrows()]

    # Rows every matcher above missed get one last chance: dense passage
    # retrieval over the encyclopedia index (dense_rag.py — a printed no-op
    # unless its index + embedding model are attached).
    miss = [i for i, r in enumerate(out) if r is None]
    if miss:
        try:
            import dense_rag
        except ImportError:
            dense_rag = None
        if dense_rag is not None:
            sub = df.iloc[miss]
            answers = (sub["response_bn"].astype(str).tolist()
                       if "response_bn" in sub.columns else [""] * len(sub))
            try:
                dense = dense_rag.build_references(
                    sub["prompt_bn"].astype(str).tolist(), answers)
                for i, ref in zip(miss, dense):
                    out[i] = ref
            except Exception as e:
                # a broken dense fallback must never sink the submission —
                # these rows just fall back to model-knowledge judging
                print(f"[rag] dense fallback errored "
                      f"({type(e).__name__}: {e}) — {len(miss)} rows judged "
                      "from model knowledge")

    kinds = Counter(v["kind"] for v in out if v is not None)
    n = sum(kinds.values())
    detail = ", ".join(f"{c} {k}" for k, c in kinds.most_common()) or "none"
    print(f"[rag] {branch}: {n}/{len(df)} rows RETRIEVED a reference "
          f"({detail}) | {len(df) - n}/{len(df)} rows NOT retrieved "
          "(judged from model knowledge alone)")
    return out


# -------------------------------------------------------------- self-test
def _selftest():
    import tempfile
    import time

    t0 = time.time()
    qa_files, law_files = find_data_files()
    assert qa_files, "no QA files found — set DATA_DIR or add filter_data/"
    bank = Bank(qa_files)
    law = LawIndex(law_files) if law_files else None
    print(f"[test] load time: {time.time() - t0:.1f}s\n")

    # 1. exact match: the first row of every file must be retrievable
    for path, qcol, acol in qa_files:
        r = pd.read_csv(path, nrows=1).iloc[0] if path.endswith(".csv") \
            else pd.read_parquet(path).iloc[0]
        hits = bank.query(r[qcol])
        assert hits and hits[0]["exact"], f"exact miss for {path}"
        keys = [a for a, _ in hits[0]["answers"]]
        assert hard(str(r[acol])) in [hard(k) for k in keys], \
            f"wrong answer for {path}"
        print(f"[test] exact  OK  {os.path.basename(path)}: "
              f"Q={clean(r[qcol])[:60]}... -> A={keys[0][:60]}")
    print()

    # 2. fuzzy match: perturbed questions must still hit the original
    t0, n_ok = time.time(), 0
    samples = [bank.qs[i] for i in range(0, len(bank.qs),
                                         max(1, len(bank.qs) // 20))][:20]
    for q in samples:
        if len(q) < 25:
            continue
        mangled = "আচ্ছা, " + q[: int(len(q) * 0.85)]  # prefix + truncated
        hits = bank.query(mangled)
        if hits and any(h["question"] == q for h in hits):
            n_ok += 1
    print(f"[test] fuzzy  OK  {n_ok}/{len([q for q in samples if len(q) >= 25])} "
          f"perturbed questions recovered "
          f"({(time.time() - t0) / max(1, len(samples)):.2f}s/query)")

    # 3. negative: unrelated question must return nothing
    assert bank.query("What is the capital of France?") == []
    assert bank.query("xyzzy плугх 12345") == []
    print("[test] negative OK  (unrelated questions -> no hits)")

    # 4. type partition: ভাবার্থ query never answered by শাব্দিক entry
    checked = 0
    for i, q in enumerate(bank.qs):
        if bank.types[i] != "ভাবার্থ" or checked >= 5:
            continue
        # prefix-only perturbation: breaks the exact key but keeps the
        # ভাবার্থ marker in the query
        for h in bank.query("আচ্ছা, " + q):
            assert "শাব্দিক" not in h["question"], f"type leak: {h['question']}"
        checked += 1
    print(f"[test] type-partition OK  ({checked} ভাবার্থ queries checked)")

    # 5. build_reference end-to-end (bank + law + rule fallbacks)
    rows = [bank.qs[0],
            "নিরামিষ যে ব্যাসবাক্যে কোন সমাস?",
            "'নর + ইন্দ্র' সন্ধিতে কী হয়?",
            "totally unrelated question with no reference at all 42"]
    law_q = None
    if law and law.acts:
        # titles already carry ", year"; pick any act the regex can resolve
        for act in law.acts.values():
            if not act["secs"]:
                continue
            sec = next(iter(act["secs"]))
            q = f"{act['title']} এর {sec} ধারা অনুযায়ী কী বিধান আছে?"
            if law.reference(q):
                law_q = q
                break
        assert law_q, "no act resolvable for law test"
        rows.insert(1, law_q)
    df = pd.DataFrame({"prompt_bn": rows})
    _CACHE.update(bank=bank, law=law)  # reuse the already-built index
    refs = build_reference(df, "selftest")
    for q, ref in zip(rows, refs):
        tag = "-" if ref is None else \
            ("fast" if ref["fast"] else "slow") + " | " + \
            ref["text"].splitlines()[0][:70]
        print(f"    {q[:55]!r:60} -> {tag}")
    assert refs[0] is not None and refs[0]["fast"], "bank ref missing"
    assert refs[-1] is None, "unrelated row must get no reference"
    assert any(r and "সমাস" in r["text"] for r in refs if r), "samas fallback"
    if law_q:
        assert refs[1] and "ধারা" in refs[1]["text"], "law ref missing"
    print("[test] build_reference OK\n")

    # 6. extensibility: a brand-new file is picked up with zero code change
    with tempfile.TemporaryDirectory() as tmp:
        pd.DataFrame({"question": ["আমার নতুন প্রশ্নটি কী?"],
                      "answer": ["এটি নতুন উত্তর।"]}
                     ).to_csv(os.path.join(tmp, "brand_new_source.csv"),
                              index=False)
        qa2, _ = find_data_files([tmp])
        b2 = Bank(qa2)
        hits = b2.query("আমার নতুন প্রশ্নটি কী?")
        assert hits and hits[0]["exact"] \
            and hits[0]["answers"][0][0] == "এটি নতুন উত্তর।"
    print("[test] extensibility OK  (new csv dropped in -> retrievable, "
          "no code change)")
    print("\nALL TESTS PASSED")


if __name__ == "__main__":
    _selftest()


Writing retrieval.py


In [5]:
%%writefile dense_rag.py
"""Dense-passage fallback: BGE-M3 + FAISS over the encyclopedia corpus.

retrieval.py steps 1-4 (exact / law / fuzzy / rule tables) only fire when
the question is close to a bank entry. Rows they all miss were judged from
model knowledge alone in v4; here they get a last-chance reference from the
index built by the rag-bangla notebook (Banglapedia + attached PDFs):

    question + candidate answer -> one claim -> BGE-M3 -> FAISS top-K
        -> [optional bge-reranker-v2-m3] -> top-N passages above MIN_SCORE

Embedding the claim (not just the question) biases retrieval toward
passages that can verify or refute that specific answer instead of
passages merely on the question's topic (see rag/retriever.py).

Every dependency is optional — with any piece missing, build_references()
returns all-None and prints once what to attach:
  * index    rag_output/index/{faiss.index,id_mapping.pkl} — attach the
             published rag_output as a dataset (auto-discovered under
             /kaggle/input) or set $RAG_DIR to the folder holding them.
  * embedder BAAI/bge-m3 — attach as a Kaggle model (any input path
             containing 'bge-m3') or set $EMBED_MODEL_DIR. Off Kaggle the
             hub id is used directly.
  * reranker bge-reranker-v2-m3 (path containing 'bge-reranker', or
             $RERANKER_DIR) — optional; FAISS order is used without it.
  * faiss    installed by the pip cell; sentence-transformers ships in
             the Kaggle image.

Runs before vllm loads (GPUs are free); run_dev/run_test call unload()
afterwards so the embedder never competes with vllm for GPU memory.

Self-test: python3 dense_rag.py
"""
import gc
import glob
import os
import pickle
import re

import numpy as np

MIN_SCORE = 0.60     # cosine floor — below this a passage is topic noise.
                     # Tune against run_dev's "[rag] ... accuracy" lines.
TOP_K = 20           # FAISS candidates (reranker input)
TOP_N = 1            # passages quoted in the prompt
PASSAGE_CHARS = 700  # per-passage cap; 3x700 bn chars fits the prompt budget
ENCODE_BATCH = 64
RERANK_BATCH = 16
USE_RERANKER = True  # skipped silently when the model is not attached
DEBUG_SAMPLES = 5    # log this many retrieved (Q, reference) pairs per
                     # branch so passage quality is inspectable; 0 = off

_CACHE = {}


def _find_index_dir():
    cands = [os.environ.get("RAG_DIR", "")]
    for base in (os.path.dirname(os.path.abspath(__file__)), os.getcwd()):
        cands += [os.path.join(base, "rag_output", "index"),
                  os.path.join(base, "rag_output")]
    cands += sorted(os.path.dirname(p) for p in glob.glob(
        "/kaggle/input/**/faiss.index", recursive=True))
    for d in cands:
        if d and os.path.isfile(os.path.join(d, "faiss.index")) \
                and os.path.isfile(os.path.join(d, "id_mapping.pkl")):
            return d
    return None


def _find_model_dir(env_key, needle, hub_id):
    d = os.environ.get(env_key)
    if d and os.path.isdir(d):
        return d
    hits = [os.path.dirname(p) for p in
            glob.glob("/kaggle/input/**/config.json", recursive=True)
            if needle in os.path.dirname(p).lower()]
    if hits:
        return min(hits, key=len)      # shallowest: skips 1_Pooling/ etc.
    # off Kaggle, sentence-transformers resolves the hub id itself
    return None if os.path.isdir("/kaggle/input") else hub_id


def _neutralize_torchcodec():
    """Route a broken torchcodec to ImportError so sentence-transformers uses
    its own supported "torchcodec not installed" path.

    Kaggle's sentence-transformers imports torchcodec (audio/video) at package
    import. Its modality_types.py already guards a MISSING torchcodec:
        try:    from torchcodec.decoders import AudioDecoder, VideoDecoder
        except (ImportError, OSError):  AudioDecoder = VideoDecoder = None
    but on the T4 image torchcodec is *installed yet broken* — its .so load
    raises RuntimeError (missing libavutil + a torch C++ ABI mismatch), which
    slips past that except and crashes the whole import. We only ever embed
    text (never audio/video), so if torchcodec won't import we install a
    meta-path finder that makes any `torchcodec[.*]` import raise ImportError.
    That is exactly the "not installed" case the library supports: it sets the
    decoders to None and runs its text path unchanged. (An earlier attempt to
    hand back a stub module instead misfired — a stub whose attributes were
    `object` made isinstance(text, AudioDecoder) true, so every input was
    misread as the 'audio' modality.)"""
    import importlib.abc
    import importlib.machinery
    import sys
    try:
        import torchcodec  # noqa: F401 — leave healthy installs alone
        return
    except Exception:
        pass

    # drop any half-initialised modules the failed import left behind
    for k in [k for k in sys.modules
              if k == "torchcodec" or k.startswith("torchcodec.")]:
        del sys.modules[k]

    class _Blocker(importlib.abc.MetaPathFinder, importlib.abc.Loader):
        def find_spec(self, name, path=None, target=None):
            if name == "torchcodec" or name.startswith("torchcodec."):
                return importlib.machinery.ModuleSpec(name, self)
            return None

        def create_module(self, spec):
            return None

        def exec_module(self, module):
            raise ImportError("torchcodec disabled (broken libs on this image)")

    sys.meta_path.insert(0, _Blocker())
    print("[dense] torchcodec unavailable — routed to the library's "
          "no-torchcodec path (text embedding does not need it)")


def _load():
    """Load index + models once. False (with a printed reason) -> no-op."""
    if "ready" in _CACHE:
        return _CACHE["ready"]
    _CACHE["ready"] = False
    idx_dir = _find_index_dir()
    if idx_dir is None:
        print("[dense] no faiss.index/id_mapping.pkl anywhere — dense "
              "fallback off (attach rag_output or set $RAG_DIR)")
        return False
    embed_dir = _find_model_dir("EMBED_MODEL_DIR", "bge-m3", "BAAI/bge-m3")
    if embed_dir is None:
        print("[dense] BGE-M3 not attached — dense fallback off "
              "(attach the model or set $EMBED_MODEL_DIR)")
        return False
    try:
        import faiss
        import torch
        _neutralize_torchcodec()
        from sentence_transformers import SentenceTransformer
    except Exception as e:
        print(f"[dense] import failed ({type(e).__name__}: {e}) — "
              "dense fallback off")
        return False
    try:
        index = faiss.read_index(os.path.join(idx_dir, "faiss.index"))
        with open(os.path.join(idx_dir, "id_mapping.pkl"), "rb") as f:
            mapping = pickle.load(f)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = SentenceTransformer(embed_dir, device=device,
                                    trust_remote_code=True)
        model.max_seq_length = 512     # claims are short; halves encode time
        if device == "cuda":
            model = model.half()
    except Exception as e:  # a broken attachment must not kill the run
        print(f"[dense] load failed ({e}) — dense fallback off")
        return False
    _CACHE.update(index=index, mapping=mapping, model=model)
    if USE_RERANKER:
        rr_dir = _find_model_dir("RERANKER_DIR", "bge-reranker",
                                 "BAAI/bge-reranker-v2-m3")
        if rr_dir is not None:
            try:
                from sentence_transformers import CrossEncoder
                _CACHE["reranker"] = CrossEncoder(
                    rr_dir, max_length=1024, device=device,
                    trust_remote_code=True)
            except Exception as e:
                print(f"[dense] reranker load failed ({e}) — FAISS order kept")
    print(f"[dense] {index.ntotal} chunks ({idx_dir}) | embedder {embed_dir}"
          f" | reranker {'on' if 'reranker' in _CACHE else 'off'}")
    _CACHE["ready"] = True
    return True


def _block(hits, mapping):
    lines = []
    for score, i in hits:
        meta = mapping[i]
        head = " — ".join(t for t in (meta.get("article_title"),
                                      meta.get("section_title")) if t)
        text = re.sub(r"\s+", " ", str(meta.get("text", ""))).strip()
        lines.append(f"- [{head}] (similarity {score:.2f}) "
                     f"{text[:PASSAGE_CHARS]}")
    return ("REFERENCE — encyclopedia passages retrieved for this Question "
            "from a Bengali knowledge corpus. They are evidence, not an "
            "answer key: use them when they cover the needed fact; "
            "otherwise judge from your own knowledge:\n" + "\n".join(lines))


def _debug_dump(questions, answers, out):
    """Print a few retrieved (question, reference) pairs to the run log so the
    dense passages can be eyeballed. Fires once per branch (ctx, then null)."""
    if not DEBUG_SAMPLES:
        return
    hit = sum(r is not None for r in out)
    print(f"[dense-debug] {hit}/{len(out)} rows got dense passages; "
          f"showing up to {DEBUG_SAMPLES}:", flush=True)
    shown = 0
    for i, ref in enumerate(out):
        if ref is None:
            continue
        block = ref["text"].replace("\n", "\n              ")
        print(f"[dense-debug] #{shown + 1}\n"
              f"       Q: {str(questions[i])[:220]}\n"
              f"       Candidate: {str(answers[i])[:180]}\n"
              f"       {block}", flush=True)
        shown += 1
        if shown >= DEBUG_SAMPLES:
            break


def build_references(questions, answers):
    """Per (question, answer): None, or {"text", "fast", "kind"} — the same
    shape retrieval._reference_for emits. fast=False: passages are evidence
    to reason over, not a direct answer-key comparison."""
    out = [None] * len(questions)
    if not questions or not _load():
        return out
    index, mapping = _CACHE["index"], _CACHE["mapping"]
    claims = []
    for q, a in zip(questions, answers):
        q, a = str(q).strip(), str(a).strip()
        claims.append(f"{q} উত্তর: {a}" if a else q)
    vecs = _CACHE["model"].encode(
        claims, batch_size=ENCODE_BATCH, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False).astype(np.float32)
    scores, ids = index.search(vecs, TOP_K)
    cands = [[(float(s), int(i)) for s, i in zip(srow, irow)
              if i >= 0 and s >= MIN_SCORE]
             for srow, irow in zip(scores, ids)]

    reranker = _CACHE.get("reranker")
    if reranker is not None:
        # one flat predict over all (claim, passage) pairs — batching across
        # rows keeps the reranker GPU-bound
        pairs = [(claims[r], str(mapping[i].get("text", "")))
                 for r, row in enumerate(cands) for _, i in row]
        if pairs:
            rr = np.asarray(reranker.predict(pairs, batch_size=RERANK_BATCH,
                                             show_progress_bar=False))
            pos = 0
            for r, row in enumerate(cands):
                chunk = rr[pos:pos + len(row)]
                pos += len(row)
                cands[r] = [row[j] for j in np.argsort(-chunk)[:TOP_N]]

    for r, row in enumerate(cands):
        if row:
            out[r] = {"text": _block(row[:TOP_N], mapping), "fast": False,
                      "kind": "dense-passage"}

    _debug_dump(questions, answers, out)
    return out


def unload():
    """Free embedder/reranker GPU memory — call before vllm loads."""
    _CACHE.clear()
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except ImportError:
        pass


if __name__ == "__main__":
    demo_q = ["বাংলাদেশের জাতীয় ফুল কোনটি?",
              "রবীন্দ্রনাথ ঠাকুর কত সালে নোবেল পুরস্কার লাভ করেন?"]
    demo_a = ["শাপলা", "১৯১৩ সালে"]
    for q, ref in zip(demo_q, build_references(demo_q, demo_a)):
        print(f"\nQ: {q}")
        print("(no reference)" if ref is None else ref["text"][:600])
    unload()


Writing dense_rag.py


In [6]:
%%writefile run_dev.py
import json
import os

import pandas as pd

import dense_rag
import retrieval
from judge import (CTX_VOTES, NULL_VOTES, choose_mapping, load_model,
                   macro_f1, run_branch)

# Above this many test rows (phase-2 scale), the dev stage is skipped: it
# costs ~40 min + a model load, and the 9 h budget belongs to the test set.
DEV_SKIP_TEST_ROWS = 3500


def dev_should_run():
    n = 0
    for f in ("test_ctx.parquet", "test_null.parquet"):
        if os.path.exists(f):
            n += len(pd.read_parquet(f, columns=["id"]))
    return n <= DEV_SKIP_TEST_ROWS, n


def rag_report(dev_ctx, ctx_pred, ctx_ref, dev_null, null_pred, null_ref):
    """How well does the judge do on rows that carried a reference block?"""
    for name, df, refs, pred in (("ctx", dev_ctx, ctx_ref, ctx_pred),
                                 ("null", dev_null, null_ref, null_pred)):
        with_ref = [(t, p) for t, p, r in
                    zip(df["label"], pred, refs) if r is not None]
        if with_ref:
            acc = sum(t == p for t, p in with_ref) / len(with_ref)
            print(f"[rag] {name}: {len(with_ref)} dev rows had references; "
                  f"judge accuracy on them {acc:.3f}")


def main():
    run, n_test = dev_should_run()
    if not run:
        print(f"[dev] test set has {n_test} rows (phase-2 scale) — skipping "
              "the dev stage to protect the 9 h budget; abstains map to 0 "
              "(the dev-tuned choice on both prior runs).")
        with open("null_map_choice.json", "w") as f:
            json.dump({"map_unknown_to": 0}, f)
        return

    dev_ctx = pd.read_parquet("dev_ctx.parquet")
    dev_null = pd.read_parquet("dev_null.parquet")
    with open("fewshot.json", encoding="utf-8") as f:
        fewshot = json.load(f)

    ctx_ref = retrieval.build_reference(dev_ctx, "ctx")
    null_ref = retrieval.build_reference(dev_null, "null")

    dense_rag.unload()  # free the embedder's GPUs before vllm loads
    llm, tokenizer = load_model()
    ctx_pred, ctx_dbg = run_branch(llm, tokenizer, dev_ctx, "ctx",
                                   fewshot["ctx"], CTX_VOTES, ctx_ref)
    null_raw, null_dbg = run_branch(llm, tokenizer, dev_null, "null",
                                    fewshot["null"], NULL_VOTES, null_ref)

    mapping, report = choose_mapping(dev_ctx["label"].tolist(), ctx_pred,
                                     dev_null["label"].tolist(), null_raw)
    print("=" * 60)
    print("DEV REPORT — read this before trusting the submission")
    print(report)
    null_pred = [mapping if v == -1 else v for v in null_raw]
    rag_report(dev_ctx, ctx_pred, ctx_ref, dev_null, null_pred, null_ref)
    y = dev_ctx["label"].tolist() + dev_null["label"].tolist()
    print(f"overall dev macro F1: {macro_f1(y, ctx_pred + null_pred):.4f}")
    print("=" * 60)
    with open("null_map_choice.json", "w") as f:
        json.dump({"map_unknown_to": mapping}, f)

    # Per-row dumps for error analysis (label vs pred, plus the model's answer tail)
    pd.concat([dev_ctx.reset_index(drop=True), ctx_dbg], axis=1) \
        .assign(pred=ctx_pred, has_ref=[r is not None for r in ctx_ref]) \
        .to_csv("dev_ctx_debug.csv", index=False)
    pd.concat([dev_null.reset_index(drop=True), null_dbg], axis=1) \
        .assign(raw=null_raw, has_ref=[r is not None for r in null_ref]) \
        .to_csv("dev_null_debug.csv", index=False)


if __name__ == "__main__":
    main()


Writing run_dev.py


In [7]:
%%writefile run_test.py
import json
import os
import time

import pandas as pd

import dense_rag
import retrieval
from judge import CTX_VOTES, NULL_VOTES, load_model, run_branch

# Wall-clock budget for full-thinking generation across both branches. The
# phase-2 rerun allows 9 h for ~5000 rows and the dev stage auto-skips on a
# test set that size, leaving ~8 h for this stage. Rows past the budget are
# judged in fast mode (~3.5 s/row) instead of timing out. Worst case (5000
# rows, zero references): 5.5 h thinking (~2900 rows) + ~2 h fast for the
# rest + ~0.6 h pip/prep/load — under the limit.
THINK_BUDGET_H = 5.5


def main():
    test_ctx = pd.read_parquet("test_ctx.parquet")
    test_null = pd.read_parquet("test_null.parquet")
    with open("fewshot.json", encoding="utf-8") as f:
        fewshot = json.load(f)

    ctx_ref = retrieval.build_reference(test_ctx, "ctx")
    null_ref = retrieval.build_reference(test_null, "null")

    dense_rag.unload()  # free the embedder's GPUs before vllm loads
    llm, tokenizer = load_model()
    deadline = time.time() + THINK_BUDGET_H * 3600
    ctx_pred, ctx_dbg = run_branch(llm, tokenizer, test_ctx, "ctx",
                                   fewshot["ctx"], CTX_VOTES, ctx_ref,
                                   deadline=deadline)
    null_raw, null_dbg = run_branch(llm, tokenizer, test_null, "null",
                                    fewshot["null"], NULL_VOTES, null_ref,
                                    deadline=deadline)

    mapping = 0
    if os.path.exists("null_map_choice.json"):
        with open("null_map_choice.json") as f:
            mapping = json.load(f)["map_unknown_to"]
    print(f"mapping abstains: -1 -> {mapping} "
          f"({sum(1 for v in null_raw if v == -1)} abstains)")
    null_pred = [mapping if v == -1 else v for v in null_raw]

    submission = pd.concat([
        pd.DataFrame({"id": test_ctx["id"], "label": ctx_pred}),
        pd.DataFrame({"id": test_null["id"], "label": null_pred}),
    ]).sort_values("id").reset_index(drop=True)

    expected_ids = sorted(pd.concat([test_ctx["id"], test_null["id"]]).tolist())
    assert submission["id"].tolist() == expected_ids, "submission ids are broken"
    submission.to_csv("submission.csv", index=False)

    print(f"submission.csv written: {len(submission)} rows")
    print(submission["label"].value_counts())

    # Per-row dumps for post-mortem analysis
    pd.concat([test_ctx.reset_index(drop=True), ctx_dbg], axis=1) \
        .assign(pred=ctx_pred, has_ref=[r is not None for r in ctx_ref]) \
        .to_csv("test_ctx_debug.csv", index=False)
    pd.concat([test_null.reset_index(drop=True), null_dbg], axis=1) \
        .assign(raw=null_raw, pred=null_pred,
                has_ref=[r is not None for r in null_ref]) \
        .to_csv("test_null_debug.csv", index=False)


if __name__ == "__main__":
    main()


Writing run_test.py


In [8]:
!python3 prep_data.py
!python3 run_test.py

data dir: /kaggle/input/competitions/bengali-hallucination
few-shot: ctx=2 null=4
dev eval: ctx=128 null=165
test:     ctx=1361 null=1155 ids 1..2516
[rag] loaded data1: 919/920 usable rows
[rag] loaded data2: 3934/3938 usable rows
[rag] loaded data21: 9011/9012 usable rows
[rag] loaded data22: 2802/2802 usable rows
[rag] loaded data23: 75032/75032 usable rows
[rag] loaded data3: 175/175 usable rows
[rag] loaded data4: 14744/14750 usable rows
[rag] loaded data5: 72909/72944 usable rows
[rag] loaded data6: 4234/4234 usable rows
[rag] loaded data7: 2503/2503 usable rows
[rag] loaded data8: 20718/20718 usable rows
[rag] bank: 193552 unique questions from 11 files
[rag] law corpus: 1083 acts
[dense] torchcodec unavailable — routed to the library's no-torchcodec path (text embedding does not need it)
[dense] 37867 chunks (/kaggle/input/notebooks/siamahmad/rag-bangla/kaggle_dataset/index) | embedder /kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3 | reranker off
[dense-debug] 

In [9]:
import os

import pandas as pd

assert os.path.exists("submission.csv"), (
    "submission.csv was not produced - a stage above crashed; read its log")
sub = pd.read_csv("submission.csv")
assert list(sub.columns) == ["id", "label"] and len(sub) > 0
print(f"OK: submission.csv with {len(sub)} rows")
print(sub["label"].value_counts())

OK: submission.csv with 2516 rows
label
1    1305
0    1211
Name: count, dtype: int64
